In [1]:
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [2]:
# load dataset
potholes = gpd.read_file('datasets/potholes_final.gpkg')
# select features
feature_cols = [
    'road_age',
    'years_since_surface',
    'Indice_PCI',
    'Indice_IRI',
    'freeze_thaw_30d',
    'freeze_thaw_60d',
    'precip_30d',
    'precip_60d',
    'MeanTemp',
    'SnowOnGround'
]
# select target
target = 'repeat'

# check missing
print(potholes[feature_cols + [target]].isnull().sum())

road_age                65290
years_since_surface    177574
Indice_PCI              37475
Indice_IRI              44282
freeze_thaw_30d             0
freeze_thaw_60d             0
precip_30d                  0
precip_60d                  0
MeanTemp                 3541
SnowOnGround           251659
repeat                      0
dtype: int64


In [3]:
# Clean data with missing values
# SnowOnGround probably months with no snow
potholes['SnowOnGround'] = potholes['SnowOnGround'].fillna(0)
# Mean Temp not many missing so just fill median
potholes['MeanTemp'] = potholes['MeanTemp'].fillna(potholes['MeanTemp'].median())
# Drop rows that dont have road condition features
df_model = potholes.dropna(subset=['road_age', 'years_since_surface', 'Indice_PCI', 'Indice_IRI'])

print(f"Rows before: {len(potholes)}")
print(f"Rows after dropping nulls: {len(df_model)}")
print(f"Kept: {len(df_model)/len(potholes):.1%}")

Rows before: 907266
Rows after dropping nulls: 692567
Kept: 76.3%


In [4]:
# Split data by date
df_model = df_model.sort_values('Date')
split_idx = int(len(df_model) *0.8)

train_set = df_model.iloc[:split_idx]
test_set = df_model.iloc[split_idx:]

X_train = train_set[feature_cols]
y_train = train_set[target]
X_test = test_set[feature_cols]
y_test = test_set[target]

print(f"Train: {len(X_train)} ({train_set['Date'].min()} to {train_set['Date'].max()})")
print(f"Test: {len(X_test)} ({test_set['Date'].min()} to {test_set['Date'].max()})")

Train: 554053 (2016-12-07 00:00:00 to 2023-03-22 00:00:00)
Test: 138514 (2023-03-22 00:00:00 to 2025-05-20 00:00:00)


In [5]:
# Model
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print("Training complete")

Training complete


In [6]:
# Predict on test set
y_pred = rf.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[71301 33377]
 [19654 14182]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.68      0.73    104678
           1       0.30      0.42      0.35     33836

    accuracy                           0.62    138514
   macro avg       0.54      0.55      0.54    138514
weighted avg       0.67      0.62      0.64    138514

